In [1]:
# ======================================
# 04 Strategy Backtest + Cohort Analysis - V4
# ======================================
#
# Purpose:
# --------
# Notebook 03 trained multiple V4 signal variants and exported scored predictions.
#
# This notebook evaluates how those predictions behave when translated into
# portfolio strategies.
#
# Key design:
# -----------
# We do NOT assume the Core-trained model must be used for Core Alpha,
# or the Convex-trained model must be used for Convex Alpha.
#
# Instead, we test each signal under each portfolio construction:
#
# Signals:
#   1. shared_baseline_v4
#   2. core_alpha_v4
#   3. convex_alpha_v4
#
# Portfolio policies:
#   1. Core Alpha:
#        Diversified top-20 rank-weighted basket.
#
#   2. Convex Alpha:
#        Concentrated top-3 rank-weighted basket.
#
# This separates:
#   signal generation
# from:
#   portfolio construction

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Paths ----
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PRICE_FILE = RAW_DIR / "prices.parquet"
BENCHMARK_PRICE_FILE = RAW_DIR / "benchmark_prices.parquet"

# ======================================
# Candidate Signal Files
# ======================================
#
# These are exported by Notebook 03.

SIGNAL_FILES = {
    "shared_baseline_v4": PROCESSED_DIR / "scored_test_shared_baseline_v4.parquet",
    "core_alpha_v4": PROCESSED_DIR / "scored_test_core_alpha_v4.parquet",
    "convex_alpha_v4": PROCESSED_DIR / "scored_test_convex_alpha_v4.parquet",
}

# ---- Load price data ----
prices = pd.read_parquet(PRICE_FILE)

prices["date"] = pd.to_datetime(prices["date"])
prices["ticker"] = prices["ticker"].astype(str).str.upper().str.strip()
prices["year_month"] = prices["date"].dt.to_period("M")

prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Prices shape:", prices.shape)

print("\nCandidate signal files:")
for name, path in SIGNAL_FILES.items():
    print(name, "->", path, "| exists:", path.exists())

Prices shape: (1895356, 11)

Candidate signal files:
shared_baseline_v4 -> /Users/neilyejjey/stock_signal_engine_v1/data/processed/scored_test_shared_baseline_v4.parquet | exists: True
core_alpha_v4 -> /Users/neilyejjey/stock_signal_engine_v1/data/processed/scored_test_core_alpha_v4.parquet | exists: True
convex_alpha_v4 -> /Users/neilyejjey/stock_signal_engine_v1/data/processed/scored_test_convex_alpha_v4.parquet | exists: True


In [2]:
# ======================================
# Monthly Stock Prices
# ======================================
#
# We evaluate portfolio selections at monthly horizons.
# For each ticker and month, we use the final available trading day.
#
# Result:
# ticker | year_month | month-end adjusted close

stock_monthly = (
    prices.sort_values("date")
    .groupby(["ticker", "year_month"])
    .tail(1)
    .reset_index(drop=True)
)

stock_monthly = stock_monthly[
    ["date", "year_month", "ticker", "adj_close"]
].copy()

stock_monthly = stock_monthly.sort_values(
    ["ticker", "year_month"]
).reset_index(drop=True)

print("Monthly stock prices shape:", stock_monthly.shape)
stock_monthly.head()

Monthly stock prices shape: (90471, 4)


,date,year_month,ticker,adj_close
0,2010-01-29,2010-01,A,17.707209
1,2010-02-26,2010-02,A,19.874031
2,2010-03-31,2010-03,A,21.724977
3,2010-04-30,2010-04,A,22.906305
4,2010-05-28,2010-05,A,20.442579


In [3]:
# ======================================
# Monthly Benchmark Prices
# ======================================
#
# We use SPY as the benchmark for excess-return comparisons.
#
# Prefer benchmark_prices.parquet from Notebook 01.
# Fall back to yfinance only if needed.

BENCHMARK_TICKER = "SPY"

if BENCHMARK_PRICE_FILE.exists():
    benchmark_prices = pd.read_parquet(BENCHMARK_PRICE_FILE)

    benchmark_prices["date"] = pd.to_datetime(benchmark_prices["date"])
    benchmark_prices["ticker"] = (
        benchmark_prices["ticker"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    benchmark = (
        benchmark_prices[benchmark_prices["ticker"] == BENCHMARK_TICKER]
        .copy()
    )

    if benchmark.empty:
        raise ValueError(
            f"{BENCHMARK_TICKER} not found in {BENCHMARK_PRICE_FILE}"
        )

    benchmark = benchmark.rename(columns={
        "adj_close": "benchmark_adj_close"
    })

    benchmark["year_month"] = benchmark["date"].dt.to_period("M")

    print(f"Loaded benchmark from: {BENCHMARK_PRICE_FILE}")

else:
    print("Benchmark file not found. Falling back to yfinance.")

    import yfinance as yf

    START_DATE = prices["date"].min().strftime("%Y-%m-%d")
    END_DATE = (prices["date"].max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    benchmark = yf.download(
        BENCHMARK_TICKER,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False,
        threads=False,
    )

    benchmark.columns = [
        col[0] if isinstance(col, tuple) else col
        for col in benchmark.columns
    ]

    benchmark = benchmark.reset_index().rename(columns={
        "Date": "date",
        "Adj Close": "benchmark_adj_close",
    })

    benchmark["date"] = pd.to_datetime(benchmark["date"])
    benchmark["year_month"] = benchmark["date"].dt.to_period("M")

benchmark_monthly = (
    benchmark.sort_values("date")
    .groupby("year_month")
    .tail(1)
    .reset_index(drop=True)
)

benchmark_monthly = benchmark_monthly[
    ["date", "year_month", "benchmark_adj_close"]
].copy()

benchmark_monthly = benchmark_monthly.sort_values(
    "year_month"
).reset_index(drop=True)

print("Monthly benchmark prices shape:", benchmark_monthly.shape)
benchmark_monthly.head()

Loaded benchmark from: /Users/neilyejjey/stock_signal_engine_v1/data/raw/benchmark_prices.parquet
Monthly benchmark prices shape: (192, 3)


,date,year_month,benchmark_adj_close
0,2010-01-29,2010-01,80.145447
1,2010-02-26,2010-02,82.645569
2,2010-03-31,2010-03,87.677010
3,2010-04-30,2010-04,89.033348
4,2010-05-28,2010-05,81.959244


In [4]:
# ======================================
# Create Forward Returns by Holding Horizon
# ======================================
#
# We compute realized future returns at multiple horizons:
#
# 1m, 3m, 6m, 12m, 24m, 36m
#
# This lets us diagnose when the signal matures.
#
# If alpha appears early:
#   the model may have shorter-horizon usefulness.
#
# If alpha only appears later:
#   the model is truly long-horizon.

HORIZONS = [1, 3, 6, 12, 24, 36]

# ---- Stock forward returns ----
stock_returns = stock_monthly.copy()

for h in HORIZONS:
    stock_returns[f"fwd_{h}m_stock_return"] = (
        stock_returns.groupby("ticker")["adj_close"].shift(-h)
        / stock_returns["adj_close"]
        - 1
    )

# ---- Benchmark forward returns ----
benchmark_returns = benchmark_monthly.copy()

for h in HORIZONS:
    benchmark_returns[f"fwd_{h}m_benchmark_return"] = (
        benchmark_returns["benchmark_adj_close"].shift(-h)
        / benchmark_returns["benchmark_adj_close"]
        - 1
    )

print("Stock returns shape:", stock_returns.shape)
print("Benchmark returns shape:", benchmark_returns.shape)

stock_returns.head()

Stock returns shape: (90471, 10)
Benchmark returns shape: (192, 9)


,date,year_month,ticker,adj_close,fwd_1m_stock_return,fwd_3m_stock_return,fwd_6m_stock_return,fwd_12m_stock_return,fwd_24m_stock_return,fwd_36m_stock_return
0,2010-01-29,2010-01,A,17.707209,0.122370,0.293615,-0.003567,0.492330,0.515163,0.613551
1,2010-02-26,2010-02,A,19.874031,0.093134,0.028608,-0.142721,0.337571,0.386522,0.331684
2,2010-03-31,2010-03,A,21.724977,0.054376,-0.173306,-0.029660,0.302123,0.297189,0.236141
3,2010-04-30,2010-04,A,22.906305,-0.107557,-0.229730,-0.040265,0.376448,0.165888,0.157586
4,2010-05-28,2010-05,A,20.442579,-0.121447,-0.166564,0.082200,0.541100,0.259322,0.422613


In [5]:
# ======================================
# Strategy Definitions
# ======================================
#
# We now evaluate two intentional strategy families.
#
# 1. Core Alpha
#    Goal:
#       Diversified, market-beating, more robust stock selection.
#
#    Candidate default:
#       top 20 names, rank-weighted.
#
# 2. Convex Alpha
#    Goal:
#       Concentrated exposure to public-market power-law winners.
#
#    Candidate default:
#       top 3 names, rank-weighted.
#
# We keep this configurable because the backtest is still telling us
# which version is most defensible.

STRATEGIES = {
    "core_alpha": {
        "top_k": 20,
        "power": 0.6,
        "description": "Diversified rank-weighted alpha basket"
    },
    "convex_alpha": {
        "top_k": 3,
        "power": 0.8,
        "description": "Concentrated right-tail winner capture basket"
    },
}

In [6]:
# ======================================
# Build Strategy Portfolios
# ======================================
#
# For each month:
#   1. Take all stocks scored by the model.
#   2. Select the highest-ranked names.
#   3. Assign rank-based weights.
#   4. Store the selected cohort.
#
# This does not yet simulate capital flowing month by month.
# It builds monthly "selection cohorts" for horizon analysis.

def build_topk_strategy(
    df,
    strategy_name,
    top_k,
    power,
    score_col="calibrated_pred"
):
    portfolios = []

    for ym, group in df.groupby("year_month"):
        group = group.dropna(subset=[score_col]).copy()

        if group.empty:
            continue

        k = min(top_k, len(group))

        # Select top-scoring names
        subset = group.nlargest(k, score_col).copy()

        # Rank-based weighting
        ranks = subset[score_col].rank(method="first")
        w = ranks ** power
        subset["weight"] = w / w.sum()

        # Metadata
        subset["strategy"] = strategy_name
        subset["cohort_month"] = ym
        subset["top_k"] = top_k
        subset["power"] = power
        subset["num_names_in_universe"] = len(group)
        subset["num_names_selected"] = k

        portfolios.append(subset)

    if not portfolios:
        return pd.DataFrame()

    return pd.concat(portfolios, ignore_index=True)


In [7]:
# ======================================
# Cohort Aging Performance
# ======================================
#
# For each strategy and horizon:
#   - Calculate one weighted return per cohort month.
#   - Compare that cohort return to SPY over the same horizon.
#   - Aggregate across cohorts.
#
# This answers:
#   "When does this strategy's edge show up?"
#
# Implementation note:
#   This version avoids groupby.apply() and uses explicit weighted columns
#   + groupby.agg(), which is cleaner and avoids pandas deprecation warnings.

def summarize_by_horizon(portfolio, strategy_name):
    rows = []

    for h in HORIZONS:
        stock_col = f"fwd_{h}m_stock_return"
        bench_col = f"fwd_{h}m_benchmark_return"
        excess_col = f"fwd_{h}m_excess_return"

        valid = portfolio.dropna(
            subset=[stock_col, bench_col, excess_col, "weight"]
        ).copy()

        if valid.empty:
            rows.append({
                "strategy": strategy_name,
                "horizon_months": h,
                "num_cohorts": 0,
                "mean_portfolio_return": np.nan,
                "median_portfolio_return": np.nan,
                "mean_benchmark_return": np.nan,
                "mean_excess_return": np.nan,
                "median_excess_return": np.nan,
                "excess_std": np.nan,
                "hit_rate": np.nan,
            })
            continue

        # Create explicit weighted return columns
        valid["weighted_stock_return"] = valid[stock_col] * valid["weight"]
        valid["weighted_excess_return"] = valid[excess_col] * valid["weight"]

        # One row per monthly cohort
        cohort_returns = (
            valid
            .groupby("cohort_month")
            .agg(
                portfolio_return=("weighted_stock_return", "sum"),
                benchmark_return=(bench_col, "first"),
                excess_return=("weighted_excess_return", "sum"),
                num_positions=("ticker", "count"),
            )
            .reset_index()
        )

        rows.append({
            "strategy": strategy_name,
            "horizon_months": h,
            "num_cohorts": len(cohort_returns),
            "mean_portfolio_return": cohort_returns["portfolio_return"].mean(),
            "median_portfolio_return": cohort_returns["portfolio_return"].median(),
            "mean_benchmark_return": cohort_returns["benchmark_return"].mean(),
            "mean_excess_return": cohort_returns["excess_return"].mean(),
            "median_excess_return": cohort_returns["excess_return"].median(),
            "excess_std": cohort_returns["excess_return"].std(),
            "hit_rate": (cohort_returns["excess_return"] > 0).mean(),
        })

    return pd.DataFrame(rows)


In [8]:
# ======================================
# Contributor Analysis
# ======================================
#
# This checks whether a strategy is broadly successful
# or carried by a small number of monster winners.
#
# Especially important for Convex Alpha.

def contributor_analysis(portfolio, horizon=36, top_n=20):
    excess_col = f"fwd_{horizon}m_excess_return"

    valid = portfolio.dropna(subset=[excess_col, "weight"]).copy()

    valid["weighted_excess_contribution"] = (
        valid[excess_col] * valid["weight"]
    )

    contributors = (
        valid.groupby("ticker")["weighted_excess_contribution"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
    )

    return contributors


In [9]:
# ======================================
# Contributor Concentration Metrics
# ======================================
#
# This quantifies whether strategy performance is broad-based
# or dependent on a few major winners.
#
# Helpful interpretation:
#   - High top_1_share_of_positive means one name is carrying the strategy.
#   - High top_3/top_5 share means performance is power-law concentrated.
#   - This is not automatically bad for Convex Alpha, but it should be explicit.

def contributor_concentration(portfolio, horizon=36):
    excess_col = f"fwd_{horizon}m_excess_return"

    valid = portfolio.dropna(subset=[excess_col, "weight"]).copy()

    if valid.empty:
        return pd.Series({
            "top_1_contribution": np.nan,
            "top_3_contribution": np.nan,
            "top_5_contribution": np.nan,
            "total_positive_contribution": np.nan,
            "top_1_share_of_positive": np.nan,
            "top_3_share_of_positive": np.nan,
            "top_5_share_of_positive": np.nan,
        })

    valid["weighted_excess_contribution"] = (
        valid[excess_col] * valid["weight"]
    )

    contrib = (
        valid.groupby("ticker")["weighted_excess_contribution"]
        .sum()
        .sort_values(ascending=False)
    )

    total_positive = contrib[contrib > 0].sum()

    top_1 = contrib.iloc[0] if len(contrib) >= 1 else np.nan
    top_3 = contrib.head(3).sum()
    top_5 = contrib.head(5).sum()

    return pd.Series({
        "top_1_contribution": top_1,
        "top_3_contribution": top_3,
        "top_5_contribution": top_5,
        "total_positive_contribution": total_positive,
        "top_1_share_of_positive": top_1 / total_positive if total_positive != 0 else np.nan,
        "top_3_share_of_positive": top_3 / total_positive if total_positive != 0 else np.nan,
        "top_5_share_of_positive": top_5 / total_positive if total_positive != 0 else np.nan,
    })


In [10]:
# ======================================
# Selection Frequency
# ======================================
#
# This shows whether a strategy repeatedly selects the same names.
#
# Repeated selection is not automatically bad.
# But if one or two names appear constantly, performance may be fragile.

def selection_frequency(portfolio, top_n=25):
    return (
        portfolio.groupby("ticker")["cohort_month"]
        .nunique()
        .sort_values(ascending=False)
        .head(top_n)
    )


In [11]:
# ======================================
# Sector Exposure
# ======================================
#
# This shows whether a strategy is implicitly concentrated in specific sectors.
#
# Output is normalized by number of cohorts, so values read as:
#   "average portfolio weight allocated to this sector per cohort"
#
# Example:
#   Technology = 0.25 means the strategy averaged ~25% Technology exposure.

def sector_exposure(portfolio):
    raw = (
        portfolio.groupby("sector")["weight"]
        .sum()
        .sort_values(ascending=False)
    )

    num_cohorts = portfolio["cohort_month"].nunique()

    if num_cohorts == 0:
        return raw * np.nan

    return raw / num_cohorts


In [12]:
# ======================================
# Ex-Contributor Stress Test
# ======================================
#
# This checks whether performance survives after removing the biggest winners.
#
# If performance collapses after removing one name,
# the strategy is highly contributor-sensitive.
#
# This is especially important for Convex Alpha.

def run_strategy_with_exclusions(
    df,
    strategy_name,
    top_k,
    power,
    excluded_tickers=None
):
    if excluded_tickers is None:
        excluded_tickers = []

    filtered = df[~df["ticker"].isin(excluded_tickers)].copy()

    p = build_topk_strategy(
        filtered,
        strategy_name=strategy_name,
        top_k=top_k,
        power=power,
    )

    s = summarize_by_horizon(p, strategy_name)
    return s


In [13]:
# ======================================
# Evaluate One Signal File
# ======================================
#
# This function takes one scored signal file and runs the full
# Core Alpha / Convex Alpha strategy analysis.
#
# It evaluates each signal under both portfolio policies.
#
# Important:
#   Stress tests are dynamic. Instead of hardcoding old winners like
#   PLTR/NFLX/META, we identify the top contributors for each signal/strategy
#   and test performance after excluding them.

def evaluate_signal_file(model_name, signal_file):
    print(f"\nEvaluating signal file: {model_name}")
    print("=" * 80)

    # ---- Load scored signals ----
    signals = pd.read_parquet(signal_file)

    signals["date"] = pd.to_datetime(signals["date"])
    signals["ticker"] = signals["ticker"].astype(str).str.upper().str.strip()
    signals["year_month"] = signals["date"].dt.to_period("M")

    signals = signals.sort_values(["date", "ticker"]).reset_index(drop=True)

    print("Signals shape:", signals.shape)

    # ---- Keep available signal columns ----
    signal_cols = [
        "date",
        "year_month",
        "ticker",
        "sector",
        "company",
        "sub_industry",

        # Model outputs
        "predicted_target",
        "predicted_excess_return",
        "calibrated_pred",
        "pred_rank",
        "pred_bucket",

        # 3Y labels from notebook 03
        "fwd_3y_stock_return",
        "fwd_3y_benchmark_return",
        "excess_fwd_3y_return",
        "target",
        "delta",
    ]

    signal_cols = [c for c in signal_cols if c in signals.columns]
    backtest_df = signals[signal_cols].copy()

    # ---- Merge stock forward returns ----
    stock_return_cols = ["ticker", "year_month"] + [
        f"fwd_{h}m_stock_return" for h in HORIZONS
    ]

    backtest_df = backtest_df.merge(
        stock_returns[stock_return_cols],
        on=["ticker", "year_month"],
        how="left",
    )

    # ---- Merge benchmark forward returns ----
    benchmark_return_cols = ["year_month"] + [
        f"fwd_{h}m_benchmark_return" for h in HORIZONS
    ]

    backtest_df = backtest_df.merge(
        benchmark_returns[benchmark_return_cols],
        on="year_month",
        how="left",
    )

    # ---- Create excess returns for each horizon ----
    for h in HORIZONS:
        backtest_df[f"fwd_{h}m_excess_return"] = (
            backtest_df[f"fwd_{h}m_stock_return"]
            - backtest_df[f"fwd_{h}m_benchmark_return"]
        )

    # ---- Missing return sanity check ----
    missing_summary = []

    for h in HORIZONS:
        missing_summary.append({
            "model_name": model_name,
            "horizon_months": h,
            "stock_return_missing_pct": backtest_df[f"fwd_{h}m_stock_return"].isna().mean(),
            "benchmark_return_missing_pct": backtest_df[f"fwd_{h}m_benchmark_return"].isna().mean(),
            "excess_return_missing_pct": backtest_df[f"fwd_{h}m_excess_return"].isna().mean(),
        })

    missing_summary = pd.DataFrame(missing_summary)

    # ---- Build strategy portfolios ----
    strategy_portfolios = {}

    for strategy_name, config in STRATEGIES.items():
        strategy_portfolios[strategy_name] = build_topk_strategy(
            backtest_df,
            strategy_name=strategy_name,
            top_k=config["top_k"],
            power=config["power"],
        )

    # ---- Weight checks ----
    weight_checks = []

    for strategy_name, portfolio in strategy_portfolios.items():
        if portfolio.empty:
            wc = {
                "count": 0,
                "mean": np.nan,
                "std": np.nan,
                "min": np.nan,
                "25%": np.nan,
                "50%": np.nan,
                "75%": np.nan,
                "max": np.nan,
            }
        else:
            wc = (
                portfolio.groupby("cohort_month")["weight"]
                .sum()
                .describe()
                .to_dict()
            )

        wc["model_name"] = model_name
        wc["strategy"] = strategy_name
        weight_checks.append(wc)

    weight_checks = pd.DataFrame(weight_checks)

    # ---- Horizon summaries ----
    horizon_summaries = []

    for strategy_name, portfolio in strategy_portfolios.items():
        summary = summarize_by_horizon(
            portfolio,
            strategy_name=strategy_name,
        )

        summary["model_name"] = model_name
        horizon_summaries.append(summary)

    horizon_summary = pd.concat(horizon_summaries, ignore_index=True)

    # ---- Top-K sensitivity ----
    topk_results = []

    for k in [3, 5, 10, 15, 20]:
        power = 0.8 if k <= 5 else 0.6

        p = build_topk_strategy(
            backtest_df,
            strategy_name=f"top_{k}",
            top_k=k,
            power=power,
        )

        s = summarize_by_horizon(p, f"top_{k}")
        s["model_name"] = model_name
        s["top_k"] = k
        s["power"] = power

        topk_results.append(s)

    topk_summary = pd.concat(topk_results, ignore_index=True)

    # ---- Contributor concentration ----
    concentration_summary = pd.DataFrame({
        strategy_name: contributor_concentration(portfolio, horizon=36)
        for strategy_name, portfolio in strategy_portfolios.items()
    }).T

    concentration_summary["model_name"] = model_name
    concentration_summary = (
        concentration_summary
        .reset_index()
        .rename(columns={"index": "strategy"})
    )

    # ---- Contributor tables ----
    contributor_tables = {}

    for strategy_name, portfolio in strategy_portfolios.items():
        contributor_tables[strategy_name] = contributor_analysis(
            portfolio,
            horizon=36,
        )

    # ---- Selection frequency tables ----
    selection_frequency_tables = {}

    for strategy_name, portfolio in strategy_portfolios.items():
        selection_frequency_tables[strategy_name] = selection_frequency(
            portfolio
        )

    # ---- Sector exposure tables ----
    sector_exposure_tables = {}

    for strategy_name, portfolio in strategy_portfolios.items():
        sector_exposure_tables[strategy_name] = sector_exposure(
            portfolio
        )

    # ---- Dynamic ex-contributor stress tests ----
    stress_results = []
    exclusion_records = []

    for strategy_name, config in STRATEGIES.items():
        contributors = contributor_tables[strategy_name]

        dynamic_exclusion_sets = {
            "none": [],
            "ex_top1_contributor": contributors.head(1).index.tolist(),
            "ex_top3_contributors": contributors.head(3).index.tolist(),
            "ex_top5_contributors": contributors.head(5).index.tolist(),
        }

        for exclusion_label, excluded in dynamic_exclusion_sets.items():
            exclusion_records.append({
                "model_name": model_name,
                "strategy": strategy_name,
                "exclusion_label": exclusion_label,
                "excluded_tickers": excluded,
            })

            s = run_strategy_with_exclusions(
                backtest_df,
                strategy_name=f"{strategy_name}_{exclusion_label}",
                top_k=config["top_k"],
                power=config["power"],
                excluded_tickers=excluded,
            )

            s["model_name"] = model_name
            s["base_strategy"] = strategy_name
            s["exclusion_label"] = exclusion_label
            s["excluded_tickers"] = ", ".join(excluded)

            stress_results.append(s)

    stress_summary = pd.concat(stress_results, ignore_index=True)
    exclusion_summary = pd.DataFrame(exclusion_records)

    return {
        "model_name": model_name,
        "signals": signals,
        "backtest_df": backtest_df,
        "missing_summary": missing_summary,
        "strategy_portfolios": strategy_portfolios,
        "weight_checks": weight_checks,
        "horizon_summary": horizon_summary,
        "topk_summary": topk_summary,
        "concentration_summary": concentration_summary,
        "contributor_tables": contributor_tables,
        "selection_frequency_tables": selection_frequency_tables,
        "sector_exposure_tables": sector_exposure_tables,
        "stress_summary": stress_summary,
        "exclusion_summary": exclusion_summary,
    }

In [14]:
# ======================================
# Run Backtests for All Candidate Signal Files
# ======================================

multi_results = {}

for model_name, signal_file in SIGNAL_FILES.items():
    if not signal_file.exists():
        print(f"Skipping {model_name}; file does not exist: {signal_file}")
        continue

    multi_results[model_name] = evaluate_signal_file(
        model_name=model_name,
        signal_file=signal_file,
    )

print("\nFinished evaluating candidate signal files.")
print("Models evaluated:", list(multi_results.keys()))


Evaluating signal file: shared_baseline_v4
Signals shape: (10993, 16)

Evaluating signal file: core_alpha_v4
Signals shape: (10993, 16)

Evaluating signal file: convex_alpha_v4
Signals shape: (10993, 16)

Finished evaluating candidate signal files.
Models evaluated: ['shared_baseline_v4', 'core_alpha_v4', 'convex_alpha_v4']


In [15]:
# ======================================
# Combine Multi-Model Backtest Results
# ======================================

if not multi_results:
    raise ValueError("No signal files were evaluated. Check SIGNAL_FILES paths.")

all_missing_summary = pd.concat(
    [result["missing_summary"] for result in multi_results.values()],
    ignore_index=True,
)

all_weight_checks = pd.concat(
    [result["weight_checks"] for result in multi_results.values()],
    ignore_index=True,
)

all_horizon_summary = pd.concat(
    [result["horizon_summary"] for result in multi_results.values()],
    ignore_index=True,
)

all_topk_summary = pd.concat(
    [result["topk_summary"] for result in multi_results.values()],
    ignore_index=True,
)

all_concentration_summary = pd.concat(
    [result["concentration_summary"] for result in multi_results.values()],
    ignore_index=True,
)

all_stress_summary = pd.concat(
    [result["stress_summary"] for result in multi_results.values()],
    ignore_index=True,
)

all_exclusion_summary = pd.concat(
    [result["exclusion_summary"] for result in multi_results.values()],
    ignore_index=True,
)

print("All missing summary:", all_missing_summary.shape)
print("All weight checks:", all_weight_checks.shape)
print("All horizon summary:", all_horizon_summary.shape)
print("All top-k summary:", all_topk_summary.shape)
print("All concentration summary:", all_concentration_summary.shape)
print("All stress summary:", all_stress_summary.shape)
print("All exclusion summary:", all_exclusion_summary.shape)

All missing summary: (18, 5)
All weight checks: (6, 10)
All horizon summary: (36, 11)
All top-k summary: (90, 13)
All concentration summary: (6, 9)
All stress summary: (144, 14)
All exclusion summary: (24, 4)


In [16]:
# ======================================
# Main Model Comparison: Core vs Convex at Key Horizons
# ======================================

main_horizons = [6, 12, 24, 36]

model_strategy_comparison = all_horizon_summary[
    all_horizon_summary["horizon_months"].isin(main_horizons)
][
    [
        "model_name",
        "strategy",
        "horizon_months",
        "num_cohorts",
        "mean_excess_return",
        "median_excess_return",
        "excess_std",
        "hit_rate",
    ]
].copy()

model_strategy_comparison = model_strategy_comparison.sort_values(
    ["strategy", "horizon_months", "model_name"]
).reset_index(drop=True)

model_strategy_comparison

,model_name,strategy,horizon_months,num_cohorts,mean_excess_return,median_excess_return,excess_std,hit_rate
0,convex_alpha_v4,convex_alpha,6,24,0.138852,0.112848,0.202071,0.750000
1,core_alpha_v4,convex_alpha,6,24,0.057418,0.017976,0.237907,0.541667
2,shared_baseline_v4,convex_alpha,6,24,0.207515,0.194723,0.217356,0.791667
3,convex_alpha_v4,convex_alpha,12,24,0.347743,0.300663,0.256821,0.958333
4,core_alpha_v4,convex_alpha,12,24,0.150229,0.136933,0.304852,0.666667
5,shared_baseline_v4,convex_alpha,12,24,0.309293,0.250776,0.332912,0.791667
6,convex_alpha_v4,convex_alpha,24,24,0.506775,0.376148,0.658729,0.708333
7,core_alpha_v4,convex_alpha,24,24,0.272471,-0.029735,0.591860,0.458333
8,shared_baseline_v4,convex_alpha,24,24,0.405939,0.012135,0.857173,0.541667
9,convex_alpha_v4,convex_alpha,36,24,1.126730,0.961332,0.925317,0.916667


In [17]:
# ======================================
# 36M Model Comparison
# ======================================

model_36m_comparison = all_horizon_summary[
    all_horizon_summary["horizon_months"] == 36
][
    [
        "model_name",
        "strategy",
        "num_cohorts",
        "mean_excess_return",
        "median_excess_return",
        "excess_std",
        "hit_rate",
    ]
].copy()

model_36m_comparison = model_36m_comparison.sort_values(
    ["strategy", "mean_excess_return", "median_excess_return"],
    ascending=[True, False, False],
).reset_index(drop=True)

model_36m_comparison

,model_name,strategy,num_cohorts,mean_excess_return,median_excess_return,excess_std,hit_rate
0,convex_alpha_v4,convex_alpha,24,1.126730,0.961332,0.925317,0.916667
1,shared_baseline_v4,convex_alpha,24,0.953768,0.745823,1.335461,0.625000
2,core_alpha_v4,convex_alpha,24,0.619078,0.120572,1.275189,0.541667
3,core_alpha_v4,core_alpha,24,1.659081,0.679054,1.586380,1.000000
4,shared_baseline_v4,core_alpha,24,1.492209,0.659799,1.632963,1.000000
5,convex_alpha_v4,core_alpha,24,1.257893,0.668962,0.971319,1.000000


In [18]:
# ======================================
# Top-K Sensitivity: 36M Multi-Model View
# ======================================

topk_36m_compare = all_topk_summary[
    all_topk_summary["horizon_months"] == 36
][
    [
        "model_name",
        "strategy",
        "top_k",
        "power",
        "num_cohorts",
        "mean_excess_return",
        "median_excess_return",
        "excess_std",
        "hit_rate",
    ]
].copy()

topk_36m_compare = topk_36m_compare.sort_values(
    ["top_k", "mean_excess_return", "median_excess_return"],
    ascending=[True, False, False],
).reset_index(drop=True)

topk_36m_compare

,model_name,strategy,top_k,power,num_cohorts,mean_excess_return,median_excess_return,excess_std,hit_rate
0,convex_alpha_v4,top_3,3,0.8,24,1.126730,0.961332,0.925317,0.916667
1,shared_baseline_v4,top_3,3,0.8,24,0.953768,0.745823,1.335461,0.625000
2,core_alpha_v4,top_3,3,0.8,24,0.619078,0.120572,1.275189,0.541667
3,core_alpha_v4,top_5,5,0.8,24,1.529219,0.798807,1.600291,0.875000
4,shared_baseline_v4,top_5,5,0.8,24,1.403500,0.761523,1.676688,0.750000
5,convex_alpha_v4,top_5,5,0.8,24,0.788221,0.965165,0.585041,0.916667
6,core_alpha_v4,top_10,10,0.6,24,1.790841,0.849207,1.705076,1.000000
7,shared_baseline_v4,top_10,10,0.6,24,1.072267,0.752175,0.814029,1.000000
8,convex_alpha_v4,top_10,10,0.6,24,0.921389,0.818425,0.535208,0.958333
9,core_alpha_v4,top_15,15,0.6,24,1.598616,0.797248,1.462544,1.000000


In [19]:
# ======================================
# Contributor Concentration Comparison
# ======================================

concentration_compare = all_concentration_summary[
    [
        "model_name",
        "strategy",
        "top_1_share_of_positive",
        "top_3_share_of_positive",
        "top_5_share_of_positive",
        "top_1_contribution",
        "top_3_contribution",
        "top_5_contribution",
        "total_positive_contribution",
    ]
].copy()

concentration_compare = concentration_compare.sort_values(
    ["strategy", "top_5_share_of_positive"]
).reset_index(drop=True)

concentration_compare

,model_name,strategy,top_1_share_of_positive,top_3_share_of_positive,top_5_share_of_positive,top_1_contribution,top_3_contribution,top_5_contribution,total_positive_contribution
0,convex_alpha_v4,convex_alpha,0.429569,0.768741,0.930573,14.314631,25.616959,31.009713,33.323248
1,shared_baseline_v4,convex_alpha,0.465114,0.769300,0.958890,14.314631,23.676458,29.511384,30.776621
2,core_alpha_v4,convex_alpha,0.628123,0.887723,0.985869,14.314631,20.230780,22.467480,22.789522
3,core_alpha_v4,core_alpha,0.269747,0.551438,0.662978,12.251919,25.046322,30.112490,45.420049
4,convex_alpha_v4,core_alpha,0.256218,0.539156,0.664515,9.458513,19.903457,24.531207,36.915931
5,shared_baseline_v4,core_alpha,0.232299,0.583606,0.722612,9.814652,24.657453,30.530484,42.250161


In [20]:
# ======================================
# 36M Dynamic Ex-Contributor Stress Test Comparison
# ======================================

stress_36m_compare = all_stress_summary[
    all_stress_summary["horizon_months"] == 36
][
    [
        "model_name",
        "base_strategy",
        "exclusion_label",
        "excluded_tickers",
        "num_cohorts",
        "mean_excess_return",
        "median_excess_return",
        "excess_std",
        "hit_rate",
    ]
].copy()

stress_36m_compare = stress_36m_compare.sort_values(
    [
        "base_strategy",
        "exclusion_label",
        "mean_excess_return",
        "median_excess_return",
    ],
    ascending=[True, True, False, False],
).reset_index(drop=True)

stress_36m_compare

,model_name,base_strategy,exclusion_label,excluded_tickers,num_cohorts,mean_excess_return,median_excess_return,excess_std,hit_rate
0,shared_baseline_v4,convex_alpha,ex_top1_contributor,NFLX,24,1.788800,0.745823,2.620299,0.625000
1,convex_alpha_v4,convex_alpha,ex_top1_contributor,NFLX,24,0.486997,0.486311,0.657466,0.750000
2,core_alpha_v4,convex_alpha,ex_top1_contributor,NFLX,24,0.341165,0.028889,1.044868,0.500000
3,core_alpha_v4,convex_alpha,ex_top3_contributors,"NFLX, DASH, AXON",24,1.311841,0.323807,2.418233,0.625000
4,shared_baseline_v4,convex_alpha,ex_top3_contributors,"NFLX, TPL, META",24,1.289654,0.142261,2.223533,0.500000
5,convex_alpha_v4,convex_alpha,ex_top3_contributors,"NFLX, META, TPL",24,0.063545,-0.223424,0.834380,0.333333
6,core_alpha_v4,convex_alpha,ex_top5_contributors,"NFLX, DASH, AXON, TPL, PLTR",24,0.713304,-0.409796,2.385594,0.333333
7,convex_alpha_v4,convex_alpha,ex_top5_contributors,"NFLX, META, TPL, FANG, VST",24,-0.164449,-0.369117,0.488207,0.333333
8,shared_baseline_v4,convex_alpha,ex_top5_contributors,"NFLX, TPL, META, NVDA, VST",24,-0.375349,-0.507043,0.442422,0.083333
9,convex_alpha_v4,convex_alpha,none,,24,1.126730,0.961332,0.925317,0.916667


In [21]:
# ======================================
# V4 Promotion Decision Table
# ======================================
#
# This table ranks signal + portfolio-policy pairings at the 36M horizon.
#
# It does not automatically decide final production use, but it gives a
# clean summary of the strongest candidates.

promotion_candidates = model_36m_comparison.copy()

# Add concentration info.
promotion_candidates = promotion_candidates.merge(
    concentration_compare[
        [
            "model_name",
            "strategy",
            "top_1_share_of_positive",
            "top_3_share_of_positive",
            "top_5_share_of_positive",
        ]
    ],
    on=["model_name", "strategy"],
    how="left",
)

# Heuristic robustness score:
#   Rewards mean/median excess + hit rate.
#   Penalizes volatility and top-1 concentration.
#
# This is not used for trading.
# It is a research summary score.
promotion_candidates["research_score"] = (
    promotion_candidates["mean_excess_return"].fillna(0)
    + promotion_candidates["median_excess_return"].fillna(0)
    + promotion_candidates["hit_rate"].fillna(0)
    - 0.25 * promotion_candidates["excess_std"].fillna(0)
    - 0.50 * promotion_candidates["top_1_share_of_positive"].fillna(0)
)

promotion_candidates = promotion_candidates.sort_values(
    ["strategy", "research_score"],
    ascending=[True, False],
).reset_index(drop=True)

best_by_strategy = (
    promotion_candidates
    .sort_values(["strategy", "research_score"], ascending=[True, False])
    .groupby("strategy")
    .head(1)
    .reset_index(drop=True)
)

print("Best V4 candidate by strategy:")
display(best_by_strategy)

promotion_candidates

Best V4 candidate by strategy:


,model_name,strategy,num_cohorts,mean_excess_return,median_excess_return,excess_std,hit_rate,top_1_share_of_positive,top_3_share_of_positive,top_5_share_of_positive,research_score
0,convex_alpha_v4,convex_alpha,24,1.126730,0.961332,0.925317,0.916667,0.429569,0.768741,0.930573,2.558614
1,core_alpha_v4,core_alpha,24,1.659081,0.679054,1.586380,1.000000,0.269747,0.551438,0.662978,2.806667


,model_name,strategy,num_cohorts,mean_excess_return,median_excess_return,excess_std,hit_rate,top_1_share_of_positive,top_3_share_of_positive,top_5_share_of_positive,research_score
0,convex_alpha_v4,convex_alpha,24,1.126730,0.961332,0.925317,0.916667,0.429569,0.768741,0.930573,2.558614
1,shared_baseline_v4,convex_alpha,24,0.953768,0.745823,1.335461,0.625000,0.465114,0.769300,0.958890,1.758169
2,core_alpha_v4,convex_alpha,24,0.619078,0.120572,1.275189,0.541667,0.628123,0.887723,0.985869,0.648458
3,core_alpha_v4,core_alpha,24,1.659081,0.679054,1.586380,1.000000,0.269747,0.551438,0.662978,2.806667
4,shared_baseline_v4,core_alpha,24,1.492209,0.659799,1.632963,1.000000,0.232299,0.583606,0.722612,2.627618
5,convex_alpha_v4,core_alpha,24,1.257893,0.668962,0.971319,1.000000,0.256218,0.539156,0.664515,2.555916


In [22]:
# ======================================
# Save V4 Multi-Model Backtest Outputs
# ======================================

V4_BACKTEST_DIR = PROCESSED_DIR / "v4_strategy_backtest"
V4_BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

all_missing_summary.to_csv(
    V4_BACKTEST_DIR / "all_missing_summary.csv",
    index=False,
)

all_weight_checks.to_csv(
    V4_BACKTEST_DIR / "all_weight_checks.csv",
    index=False,
)

all_horizon_summary.to_csv(
    V4_BACKTEST_DIR / "all_horizon_summary.csv",
    index=False,
)

all_topk_summary.to_csv(
    V4_BACKTEST_DIR / "all_topk_summary.csv",
    index=False,
)

all_concentration_summary.to_csv(
    V4_BACKTEST_DIR / "all_concentration_summary.csv",
    index=False,
)

all_stress_summary.to_csv(
    V4_BACKTEST_DIR / "all_stress_summary.csv",
    index=False,
)

all_exclusion_summary.to_csv(
    V4_BACKTEST_DIR / "all_exclusion_summary.csv",
    index=False,
)

model_strategy_comparison.to_csv(
    V4_BACKTEST_DIR / "model_strategy_comparison.csv",
    index=False,
)

model_36m_comparison.to_csv(
    V4_BACKTEST_DIR / "model_36m_comparison.csv",
    index=False,
)

topk_36m_compare.to_csv(
    V4_BACKTEST_DIR / "topk_36m_compare.csv",
    index=False,
)

concentration_compare.to_csv(
    V4_BACKTEST_DIR / "concentration_compare.csv",
    index=False,
)

stress_36m_compare.to_csv(
    V4_BACKTEST_DIR / "stress_36m_compare.csv",
    index=False,
)

promotion_candidates.to_csv(
    V4_BACKTEST_DIR / "promotion_candidates.csv",
    index=False,
)

best_by_strategy.to_csv(
    V4_BACKTEST_DIR / "best_by_strategy.csv",
    index=False,
)

print("Saved V4 strategy backtest outputs to:", V4_BACKTEST_DIR)

Saved V4 strategy backtest outputs to: /Users/neilyejjey/stock_signal_engine_v1/data/processed/v4_strategy_backtest


In [23]:
# ======================================
# Optional HTML Export
# ======================================

!jupyter nbconvert --to html 04_strategy_backtest_and_cohort_analysis.ipynb

[NbConvertApp] Converting notebook 04_strategy_backtest_and_cohort_analysis.ipynb to html
[NbConvertApp] Writing 427077 bytes to 04_strategy_backtest_and_cohort_analysis.html


In [24]:
from IPython.display import FileLink

FileLink("04_strategy_backtest_and_cohort_analysis.html")

/Users/neilyejjey/stock_signal_engine_v1/notebooks/04_strategy_backtest_and_cohort_analysis.html